## Стекинг

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Загрузка данных
data = pd.read_csv('https://code.s3.yandex.net/datasets/ds_s16t4_ames_housing_dataset.csv')

X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = [col for col in X.columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

# Базовые модели
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=3,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

# Оборачиваем модели в пайплайны
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf)
])

gbr_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", gbr)
])

base_estimators = [
    ("rf", rf_pipe),
    ("gbr", gbr_pipe)
]

#  Метамодель
meta_model = DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=30,
    random_state=42
)

# Стекинг
stacking_reg = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# Обучение + расчёт метрик
stacking_reg.fit(X_train, y_train)

y_pred = stacking_reg.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"RMSE стекинга: {rmse:.2f}")
print(f"R^2 стекинга:  {r2:.4f}")
